<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad 
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad 
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [203]:
import pathlib

def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    A = []
    B = []

    with open("system.txt", "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip empty lines

            # Example line: 2x + 3y - z = 5
            lhs, rhs = line.split("=")
            rhs = rhs.strip()
            B.append(float(rhs))

            # Initialize coefficients
            coeffs = {"x": 0.0, "y": 0.0, "z": 0.0}

            # Normalize by adding + before negative signs for splitting
            lhs = lhs.replace("-", "+-")
            terms = lhs.split("+")

            for term in terms:
                term = term.strip()
                if not term:
                    continue
                # Extract coefficient and variable
                for var in coeffs.keys():
                    if var in term:
                        num = term.replace(var, "").strip()
                        if num in ["", "+"]:
                            num = 1
                        elif num == "-":
                            num = -1
                        coeffs[var] = float(num)
                        break

            A.append([coeffs["x"], coeffs["y"], coeffs["z"]])

    return A, B


# Example usage
A, B = load_system(pathlib.Path("system.txt"))
print(f"{A=} {B=}")


A=[[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]] B=[4.0, 4.0, 6.0]


### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [204]:
def determinant(matrix: list[list[float]]) -> float:
    a11, a12, a13 = matrix[0]
    a21, a22, a23 = matrix[1]
    a31, a32, a33 = matrix[2]

    det = a11 * (a22 * a33 - a23 * a32) - a12 * (a21 * a33 - a23 * a31) + a13 * (a21 * a32 - a22 * a31)
    return det

print(f"{determinant(A)=}")

determinant(A)=14.0


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [205]:
def trace(matrix: list[list[float]]) -> float:
    tr = matrix[0][0] + matrix[1][1] + matrix[2][2]
    return tr

print(f"{trace(A)=}")

trace(A)=3.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [206]:
def norm(vector: list[float]) -> float:
    total = 0.0
    for value in vector:
        total += value ** 2
    return total

print(f"{norm(B)=}")

norm(B)=68.0


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [207]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    rows = len(matrix)
    cols = len(matrix[0])
    transposed = [[0.0 for _ in range(rows)] for _ in range(cols)]

    for i in range(rows):
        for j in range(cols):
            transposed[j][i] = matrix[i][j]

    return transposed

print(f"{transpose(A)=}")

transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [208]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    result = []
    for row in matrix:
        total = 0.0
        for i in range(len(vector)):
            total += row[i] * vector[i]
        result.append(total)
    return result

print(f"{multiply(A, B)=}")

multiply(A, B)=[14.0, 24.0, 28.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [209]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    def determinant(m: list[list[float]]) -> float:
        a11, a12, a13 = m[0]
        a21, a22, a23 = m[1]
        a31, a32, a33 = m[2]
        return (a11 * (a22 * a33 - a23 * a32)
            - a12 * (a21 * a33 - a23 * a31)
            + a13 * (a21 * a32 - a22 * a31)
                )

    detA = determinant(matrix)
    if detA == 0:
        raise ValueError("The system has no unique solution (det(A) = 0).")

    matrices = []
    for col in range(3):
        temp = [row[:] for row in matrix]
        for i in range(3):
            temp[i][col] = vector[i]
        matrices.append(temp)

    # Compute determinants
    dets = [determinant(m) for m in matrices]

    # Apply Cramer's rule
    x = dets[0] / detA
    y = dets[1] / detA
    z = dets[2] / detA

    return [x, y, z]

print("A =", A)
print("B =", B)
print(f"{solve_cramer(A, B)=}")


A = [[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]]
B = [4.0, 4.0, 6.0]
solve_cramer(A, B)=[1.0, 1.0, 1.0]


### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [210]:
from typing import List

def determinant(m: List[List[float]]) -> float:
    """Compute determinant for 2x2 or 3x3 matrix."""
    n = len(m)
    if n == 2:
        # 2x2 determinant: [[a,b],[c,d]] -> ad - bc
        return m[0][0] * m[1][1] - m[0][1] * m[1][0]
    if n == 3:
        a11, a12, a13 = m[0]
        a21, a22, a23 = m[1]
        a31, a32, a33 = m[2]
        return (
            a11 * (a22 * a33 - a23 * a32)
            - a12 * (a21 * a33 - a23 * a31)
            + a13 * (a21 * a32 - a22 * a31)
        )
    raise ValueError("determinant: only 2x2 and 3x3 matrices supported")


def minor(matrix: List[List[float]], i: int, j: int) -> List[List[float]]:
    """Return the 2x2 minor matrix obtained by removing row i and column j."""
    return [
        [matrix[row][col] for col in range(3) if col != j]
        for row in range(3)
        if row != i
    ]


def cofactor(matrix: List[List[float]]) -> List[List[float]]:
    """Compute the 3x3 cofactor matrix."""
    cof = [[0.0 for _ in range(3)] for _ in range(3)]
    for i in range(3):
        for j in range(3):
            sign = (-1) ** (i + j)
            # minor(matrix, i, j) is 2x2, determinant handles that now
            cof[i][j] = sign * determinant(minor(matrix, i, j))
    return cof


def adjoint(matrix: List[List[float]]) -> List[List[float]]:
    """Return the adjoint (transpose of cofactor matrix)."""
    cof = cofactor(matrix)
    # transpose cofactor
    adj = [[cof[j][i] for j in range(3)] for i in range(3)]
    return adj


def multiply(matrix: List[List[float]], vector: List[float]) -> List[float]:
    """Multiply 3x3 matrix by 3x1 vector."""
    return [sum(matrix[i][j] * vector[j] for j in range(3)) for i in range(3)]


def solve(matrix: List[List[float]], vector: List[float]) -> List[float]:
    """Solve A·X = B using adjugate/inverse method."""
    detA = determinant(matrix)
    if detA == 0:
        raise ValueError("Matrix is singular — no unique solution")

    adj = adjoint(matrix)
    inv = [[adj[i][j] / detA for j in range(3)] for i in range(3)]
    return multiply(inv, vector)


# Example (should produce [1.0, 1.0, 1.0])
A = [
    [2.0, 3.0, -1.0],
    [1.0, -1.0, 4.0],
    [3.0, 1.0, 2.0],
]
B = [4.0, 4.0, 6.0]

print(f"A = {A}")
print(f"B = {B}")
print(f"det(A) = {determinant(A)}")
print(f"solve(A, B) = {solve(A, B)}")


A = [[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]]
B = [4.0, 4.0, 6.0]
det(A) = 14.0
solve(A, B) = [1.0000000000000002, 0.9999999999999996, 1.0]
